# Notebook 1 — Prepare Documents

This notebook loads the raw data from HuggingFace, groups documents by query,
randomly selects one target document per query, and saves the result as `selected_docs.json`.

**Output:** `data/{domain}/selected_docs.json` for each domain.

**Run this notebook once before running Notebook 2 (Manipulation).**

**Important:** Set `DOMAINS` to control which domains to process.

## Setup

In [1]:
import json
import os
import random
from collections import defaultdict
from datasets import load_dataset

# Fix random seed for reproducibility
SEED = 42
random.seed(SEED)

# Paths
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))
data_dir = os.path.join(project_root, "data")

print(f"Project root: {project_root}")
print(f"Data directory: {data_dir}")

Project root: /Users/leonardrampf/Library/CloudStorage/OneDrive-Personal/Dokumente/Universität/Nova SBE/Work Project/seo-geo-generative-search-experiment
Data directory: /Users/leonardrampf/Library/CloudStorage/OneDrive-Personal/Dokumente/Universität/Nova SBE/Work Project/seo-geo-generative-search-experiment/data


## Parameters

Set `DOMAINS` to control which domains to process.
For a full run across all domains, set `DOMAINS = ALL_DOMAINS`.

In [2]:
# HuggingFace dataset
# Note: domains are splits, not configs
DATASET_PATH = "parameterlab/c-seo-bench"
ALL_DOMAINS = ["retail", "videogames", "books", "news", "web", "debate"]

# <- CHANGE THIS to control which domains to process
# For full run:  DOMAINS = ALL_DOMAINS
# For retail only: DOMAINS = ["retail"]
DOMAINS = ["retail"]

print(f"Domains to process: {DOMAINS}")

Domains to process: ['retail']


## Inspect Data Structure

In [3]:
# Quick inspection before processing
# Note: split=domain (not split='test')
inspect_domain = DOMAINS[0]
print(f"Loading {inspect_domain} from HuggingFace for inspection...")
ds = load_dataset(DATASET_PATH, split=inspect_domain)
df = ds.to_pandas()

print(f"\nColumns: {list(df.columns)}")
print(f"Total rows: {len(df)}")
print(f"Unique queries: {df['query_id'].nunique()}")

docs_per_query = df.groupby('query_id').size()
print(f"Docs per query: min={docs_per_query.min()}, max={docs_per_query.max()}, mean={docs_per_query.mean():.1f}")

print(f"\nExample query: {df['query'].iloc[0]}")
print(f"Example document (first 300 chars):\n{df['document'].iloc[0][:300]}")

Loading retail from HuggingFace for inspection...



Columns: ['query_id', 'query', 'document']
Total rows: 5000
Unique queries: 500
Docs per query: min=10, max=10, mean=10.0

Example query: holidaytraditions
Example document (first 300 chars):
Name: Graduation Ornament 2021 Guy – Class of 2021 Ornament – Personalized Christmas Ornaments – School, Teacher Ornaments – Unique Graduation Gift for Him – Polyresin Graduation Decorations 2021
Description:
List of features:
Premium Quality Polyresin Graduation Ornaments – Unlike other Holiday déc


## Helper Functions

In [4]:
def load_and_group(dataset_path, domain):
    """
    Load a domain from HuggingFace and group documents by query_id.
    Note: domain is used as split name.
    """
    print(f"  Loading {domain}...")
    ds = load_dataset(dataset_path, split=domain)
    df = ds.to_pandas()

    query2data = defaultdict(lambda: {"query": "", "list_docs": []})
    for _, row in df.iterrows():
        qid = str(row["query_id"])
        query2data[qid]["query"] = row["query"]
        query2data[qid]["list_docs"].append(row["document"])

    print(f"  -> {len(query2data)} queries loaded")
    return dict(query2data)


def sample_target_docs(query2data, n_queries=None, seed=42):
    """
    For each query, randomly select one target document.

    Output format:
    {
        "0": {
            "query_id": "...",
            "query": "...",
            "list_docs": [...],     # all documents for this query
            "target_doc_idx": 3,    # index of the selected target doc
            "doc": "..."            # the target document text
        },
        "1": { ... }
    }
    """
    random.seed(seed)

    query_ids = list(query2data.keys())

    if n_queries is not None:
        query_ids = random.sample(query_ids, min(n_queries, len(query_ids)))
        print(f"  -> Sampled {len(query_ids)} queries")

    selected_docs = {}
    for query_idx, query_id in enumerate(query_ids):
        data = query2data[query_id]
        list_docs = data["list_docs"]

        # Randomly select one target document
        target_doc_idx = random.randint(0, len(list_docs) - 1)

        selected_docs[str(query_idx)] = {
            "query_id": query_id,
            "query": data["query"],
            "list_docs": list_docs,
            "target_doc_idx": target_doc_idx,
            "doc": list_docs[target_doc_idx]
        }

    return selected_docs

## Process Domains and Save

In [5]:
for domain in DOMAINS:
    print(f"\n{'='*50}")
    print(f"Processing: {domain}")
    print(f"{'='*50}")

    # Load and group by query
    query2data = load_and_group(DATASET_PATH, domain)

    # Sample one target doc per query
    selected_docs = sample_target_docs(query2data, seed=SEED)

    # Save to data/{domain}/selected_docs.json
    output_folder = os.path.join(data_dir, domain)
    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, "selected_docs.json")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(selected_docs, f, indent=4, ensure_ascii=False)

    print(f"  -> Saved {len(selected_docs)} queries to {output_path}")

print("\nDone!")


Processing: retail
  Loading retail...
  -> 500 queries loaded
  -> Saved 500 queries to /Users/leonardrampf/Library/CloudStorage/OneDrive-Personal/Dokumente/Universität/Nova SBE/Work Project/seo-geo-generative-search-experiment/data/retail/selected_docs.json

Done!


## Inspect Output

In [6]:
# Inspect the saved selected_docs.json for the first processed domain
inspect_domain = DOMAINS[0]
output_path = os.path.join(data_dir, inspect_domain, "selected_docs.json")

with open(output_path, "r") as f:
    selected_docs = json.load(f)

print(f"Domain: {inspect_domain}")
print(f"Total queries saved: {len(selected_docs)}")

print(f"\nExample entry (query 0):")
example = selected_docs["0"]
print(f"  query_id:       {example['query_id']}")
print(f"  query:          {example['query']}")
print(f"  total docs:     {len(example['list_docs'])}")
print(f"  target_doc_idx: {example['target_doc_idx']}")
print(f"  doc (first 300 chars):\n{example['doc'][:300]}")

Domain: retail
Total queries saved: 500

Example entry (query 0):
  query_id:       50881
  query:          holidaytraditions
  total docs:     10
  target_doc_idx: 1
  doc (first 300 chars):
Name: Kids Christmas Ornaments 2021 – Personalized Garbage Truck Ornaments for Christmas Tree – Garbage Truck Ornament – Fun Ornaments for Teens & Kids 2,3,4,5,6,7,8,9 - Garbage Truck Christmas Ornament
Description:
List of features:
WHEN IT COMES TO YOUR Personalized KIDS and TEEN ornaments, you wa


## Summary — Queries per Domain

In [7]:
# Overview of all processed domains
print(f"{'Domain':12} | {'Queries':>8}")
print("-" * 25)

for domain in ALL_DOMAINS:
    path = os.path.join(data_dir, domain, "selected_docs.json")
    if not os.path.exists(path):
        print(f"{domain:12} | not yet processed")
        continue
    with open(path, "r") as f:
        docs = json.load(f)
    print(f"{domain:12} | {len(docs):>8}")

Domain       |  Queries
-------------------------
retail       |      500
videogames   | not yet processed
books        | not yet processed
news         | not yet processed
web          | not yet processed
debate       | not yet processed
